# 02 · Clustering candidate experiments (T05–T07)

All 89 base candidates on the same 1,333 eligible days and the same six-dimensional log1p+StandardScaler space:

- **E01 K-Means**: k=2..8 × seeds 100..109 → 70 runs
- **E02 Ward**: ONE euclidean linkage tree cut at k=2..8 → 7 cuts
- **E03 DBSCAN**: m=5/10/20 × eps quantiles 0.70/0.80/0.90/0.95 of the self-inclusive m-th NN distance → 12 settings

This notebook reads the authoritative run outputs; core logic lives in `src/air_quality_clustering`. **No representative k is chosen here** — selection with stability is T08.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

def latest_run(stage):
    runs = []
    for d in (ROOT / 'results' / 'runs').iterdir():
        mp = d / 'manifest.json'
        if not mp.is_file():
            continue
        m = json.loads(mp.read_text(encoding='utf-8'))
        done = m.get('stages_completed') or m.get('stage_summary', {}).get('stages_completed', [])
        if stage in done and m.get('status') in ('completed', 'completed_with_warnings'):
            runs.append((d.name, d, m))
    if not runs:
        raise RuntimeError(f'no completed run contains stage {stage}; run scripts/run_project.py first')
    return max(runs, key=lambda t: t[0])

run_name, RUN, manifest = latest_run('cluster')
print('run_id:', run_name)
experiments = pd.read_csv(RUN / 'experiments.csv')
print('candidate rows:', len(experiments))
experiments.groupby(['algorithm', 'status']).size()

## 1. K-Means (E01): metrics vs k and seed stability

Inertia decreases with k by construction — it is an elbow diagnostic, not a selection score. Silhouette/DB medians summarize 10 seeds; pairwise ARI (45 pairs per k) measures initialization sensitivity only.

In [ ]:
km = experiments[experiments.algorithm == 'kmeans']
km.groupby('k')[['silhouette', 'davies_bouldin', 'calinski_harabasz']].median().round(4)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F05_kmeans_metrics.png')))
pd.read_csv(RUN / 'tables' / 'kmeans_seed_ari.csv').round(4)

## 2. Ward (E02): one tree, seven cuts

The dendrogram below is drawn from the SAME linkage matrix that produced the labels (`artifacts/ward_linkage.npy`). Truncation is display-only.

In [ ]:
experiments[experiments.algorithm == 'ward'][['k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'min_cluster_size']].round(4)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F06_dendrogram.png')))

## 3. DBSCAN (E03): density assumption probe

Noise (-1) is never a pollution cluster. Internal metrics are computed on the non-noise subset only (`n_scored` differs from 1,333) and must not be ranked against full-sample K-Means/Ward scores. Single-cluster outcomes keep null metrics with a reason.

In [ ]:
db = experiments[experiments.algorithm == 'dbscan']
db[['experiment_id', 'min_samples', 'eps', 'n_clusters', 'noise_fraction', 'n_scored', 'silhouette', 'metric_reason', 'status']].round(4)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F07_dbscan.png')))

## 4. Same-k agreement: algorithm difference vs cluster-count difference

ARI between K-Means(seed=100) and Ward at equal k. Low values at some k mean the two geometries organize the same days differently — a finding to interpret in T09, not an error.

In [ ]:
pd.read_csv(RUN / 'tables' / 'same_k_ari.csv').round(4)

## 5. What is NOT decided here

- No representative k (T08: min-size rule → silhouette shortlist → 20 block-resample ARI).
- No cluster names, no season/weather interpretation (only after the selection is frozen).
- DBSCAN degenerate settings are retained as evidence about the density assumption.